In [1]:
import pandas as pd
import numpy as np
from glob import glob

In [2]:
filepath_pattern = 'Data/AQI_Pollutant/*.csv'
file_list = sorted(glob(filepath_pattern))
dataframes = []
cols_to_keep = ['Date', 'Daily Mean PM2.5 Concentration', 'Daily AQI Value', 'Site Latitude', 'Site Longitude', 'Local Site Name']


In [3]:
for file in file_list:
    df = pd.read_csv(file)
    df = df.loc[:, df.nunique(dropna=True) > 1]
    df = df[cols_to_keep]
    df['Date'] = pd.to_datetime(df['Date'])
    dataframes.append(df)

merged_df = pd.concat(dataframes, ignore_index=True)
temp = merged_df.groupby(['Site Latitude', 'Site Longitude'])[['Local Site Name']].count()
coords_to_drop = temp[temp['Local Site Name'] == 0].index

merged_df = merged_df[~merged_df.set_index(['Site Latitude', 'Site Longitude']).index.isin(coords_to_drop)]
duplicates = merged_df.duplicated(keep='first')
merged_df = merged_df[~duplicates]
merged_df['Date'] = pd.to_datetime(merged_df['Date'])
merged_df.head();

In [4]:
merged_df = (
    merged_df.sort_values(by='Date')
    .groupby(['Site Latitude', 'Site Longitude', 'Date'], as_index=False)
    .first()
)

merged_df.head()

,Site Latitude,Site Longitude,Date,Daily Mean PM2.5 Concentration,Daily AQI Value,Local Site Name
0,32.542679,-117.030749,2016-06-07,6.2,34,San Ysidro
1,32.542679,-117.030749,2016-06-08,8.7,48,San Ysidro
2,32.542679,-117.030749,2016-06-09,9.5,52,San Ysidro
3,32.542679,-117.030749,2016-06-10,9.9,52,San Ysidro
4,32.542679,-117.030749,2016-06-11,7.4,41,San Ysidro


In [5]:
grouped = (
    merged_df.groupby(['Site Latitude', 'Site Longitude'])
    .size()
    .reset_index(name='Count')
    .sort_values('Count', ascending=False)
)
top_15_coords = grouped.head(20)[['Site Latitude', 'Site Longitude']]


merged_df_top_15 = merged_df.merge(top_15_coords, on=['Site Latitude', 'Site Longitude'])
merged_df_top_15 = merged_df_top_15.sort_values(by=['Local Site Name', 'Date'], ascending=True).reset_index(drop=True)
merged_df_top_15.head()

,Site Latitude,Site Longitude,Date,Daily Mean PM2.5 Concentration,Daily AQI Value,Local Site Name
0,33.83062,-117.93845,2014-01-01,46.5,128,Anaheim
1,33.83062,-117.93845,2014-01-02,38.8,109,Anaheim
2,33.83062,-117.93845,2014-01-03,29.7,89,Anaheim
3,33.83062,-117.93845,2014-01-04,29.9,90,Anaheim
4,33.83062,-117.93845,2014-01-05,16.6,65,Anaheim


In [6]:
merged_df_top_15 = merged_df_top_15.rename(columns={
    'Site Latitude': 'Latitude',
    'Site Longitude': 'Longitude',
    'Daily Mean PM2.5 Concentration': 'PM2.5',
    'Daily AQI Value': 'AQI',
    'Local Site Name': 'Name'
})

merged_df_top_15.head()

,Latitude,Longitude,Date,PM2.5,AQI,Name
0,33.83062,-117.93845,2014-01-01,46.5,128,Anaheim
1,33.83062,-117.93845,2014-01-02,38.8,109,Anaheim
2,33.83062,-117.93845,2014-01-03,29.7,89,Anaheim
3,33.83062,-117.93845,2014-01-04,29.9,90,Anaheim
4,33.83062,-117.93845,2014-01-05,16.6,65,Anaheim


In [7]:
merged_df_top_15.set_index('Date', inplace=True)
merged_df_top_15.to_csv('Data/top_15_cities_pollution_data.csv', index='Date')

In [8]:
city_locations = merged_df_top_15.groupby(['Latitude', 'Longitude', 'Name']).size().reset_index(name='Count')
city_locations[['Latitude', 'Longitude', 'Name']]

,Latitude,Longitude,Name
0,32.676180,-115.483070,Calexico-Ethel Street
1,33.583018,-117.072202,Temecula
2,33.830620,-117.938450,Anaheim
3,33.996360,-117.492400,Mira Loma (Van Buren)
4,33.999580,-117.416010,Rubidoux
5,34.066590,-118.226880,Los Angeles-North Main Street
6,34.210169,-118.870509,Thousand Oaks
7,34.276316,-118.683685,Simi Valley-Cochran Street
8,34.448060,-119.231300,Ojai - East Ojai Ave
9,35.020830,-120.563880,Mesa2
